### 1.(1)

$$
\epsilon = log(y) - x^T\beta \sim N(0, \sigma^2)\\
y = exp(x^T\beta + \epsilon) \sim logN(x^T\beta, \sigma^2)\\
E(y) = exp(x^T\beta + \sigma^2/2)\\
$$

### 1.(2.1)

$$
\begin{aligned}
\frac{\partial Q(\beta)}{\partial \beta} &= 2(\frac{\partial (y-X\beta)}{\partial \beta})^T(y-X\beta) + 2\lambda \beta\\
&= -2X^T(y-X\beta) + 2\lambda \beta\\
\end{aligned}\\
\begin{aligned}
-2X^T(y-X\hat{\beta}_{ridge}) + 2\lambda \hat{\beta}_{ridge} &= 0\\
(X^TX + \lambda I_p)\hat{\beta}_{ridge} &= X^Ty\\
\hat{\beta}_{ridge} &= (X^TX + \lambda I_p)^{-1}X^Ty
\end{aligned}
$$

### 1.(2.2)

$$
\begin{aligned}
E(\hat{\beta}_{ridge}) - \beta &= E((X^TX + \lambda I_p)^{-1}X^Ty) - \beta\\
&= (X^TX + \lambda I_p)^{-1}X^TE(y) - \beta\\
&= (X^TX + \lambda I_p)^{-1}X^TX\beta - \beta\\
&= ((X^TX + \lambda I_p)^{-1}X^TX - I_p) \beta\\
&= -\lambda (X^TX + \lambda I_p)^{-1} \beta\\
\end{aligned}\\
$$

$\lambda = 0$时，$Bias(\hat{\beta}_{ridge}) = 0$对所有$\beta$都成立  
特别地，$\beta = 0$时，$Bias(\hat{\beta}_{ridge}) = 0$恒成立

### 1.(2.3)

$\lambda = 0$时
$$
\begin{aligned}
MSE(\hat{\beta}_{ridge}) &= E[||\hat{\beta}_{ridge}-\beta||^2] \\
&= E[||\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}]+E[\hat{\beta}_{ridge}]-\beta||^2] \\
&= E[||\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}]||^2 + ||E[\hat{\beta}_{ridge}]-\beta||^2 + 2(\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}])^T(E[\hat{\beta}_{ridge}]-\beta)] \\
&= E[||\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}]||^2 + ||E[\hat{\beta}_{ridge}]-\beta||^2] + 2E[(\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}])^T] (E[\hat{\beta}_{ridge}]-\beta) \\
&= E[||\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}]||^2] + ||E[\hat{\beta}_{ridge}]-\beta||^2 \\
&= E[(\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}])^T(\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}])] + ||Bias(\hat{\beta}_{ridge})||^2 \\
&= tr(E[(\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}])(\hat{\beta}_{ridge}-E[\hat{\beta}_{ridge}])^T]) + ||Bias(\hat{\beta}_{ridge})||^2 \\
&= tr(Cov(\hat{\beta}_{ridge})) + ||Bias(\hat{\beta}_{ridge})||^2 \\
&= \sigma^2tr((X^TX + \lambda I_p)^{-1}X^T X (X^TX + \lambda I_p)^{-1}) + \lambda^2 \beta^T (X^TX + \lambda I_p)^{-2} \beta\\
\end{aligned}\\
\begin{aligned}
\frac{\partial MSE(\hat{\beta}_{ridge})}{\partial \lambda} &= \sigma^2 tr(\frac{\partial X^T X (X^TX + \lambda I_p)^{-2}}{\partial \lambda}) + 2\lambda \beta^T (X^TX + \lambda I_p)^{-2} \beta + \lambda^2 \beta^T \frac{\partial (X^TX + \lambda I_p)^{-2}}{\partial \lambda} \beta\\
&= \sigma^2 tr(X^T X \frac{\partial (X^TX + \lambda I_p)^{-2}}{\partial \lambda}) + 2\lambda \beta^T (X^TX + \lambda I_p)^{-2} \beta + \lambda^2 \beta^T (-2(X^TX + \lambda I_p)^{-3}) \beta\\
&= -2\sigma^2 tr(X^T X (X^TX + \lambda I_p)^{-3}) + 2\lambda \beta^T (X^TX + \lambda I_p)^{-2} \beta - 2\lambda^2 \beta^T (X^TX + \lambda I_p)^{-3} \beta\\
\end{aligned}\\
\begin{aligned}
\frac{\partial MSE(\hat{\beta}_{ridge})}{\partial \lambda}|_{\lambda=0} &= -2\sigma^2 tr(X^T X (X^TX)^{-3})\\
&= -2\sigma^2 tr((X^TX)^{-2}) \\
&< 0 \quad X^TX\text{为正定矩阵，}(X^TX)^{-2}\text{迹大于零}\\
\end{aligned}
$$
故存在$\lambda > 0$，使得$MSE(\hat{\beta}_{ridge}) < MSE(\hat{\beta}_{ridge,0})$。

$\beta =0$时， $\hat{\beta}_{ridge,0}$ 非唯一估计，$\lambda$ 趋于正无穷时，$\hat{\beta}$趋于0，使 $MSE(\hat{\beta}_{ridge})$ 趋于0。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import RocCurveDisplay, roc_auc_score

sample_data = pd.read_csv("./sampledata.csv")
pred_data = pd.read_csv("./preddata.csv")

In [ ]:
cols = ['tenure', 'expense', 'degree', 'tightness', 'entropy', 'chgexpense', 'chgdegree']

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(12, 10))
axes = axes.flatten()

for i, f in enumerate(cols):
    sample_data.boxplot(column=f, by='churn', ax=axes[i])
    axes[i].set_title(f)
    axes[i].set_xlabel('churn')
    axes[i].set_ylabel(f)
sample_data.boxplot(column='churn', ax=axes[-1])
axes[-1].set_title('churn')

plt.suptitle('Boxplots')
plt.tight_layout()
plt.show()


In [ ]:
log_cols_o = ["tenure", "expense", "degree", "tightness"] # 对此四列进行对数化
log_cols = ["tenure_log", "expense_log", "degree_log", "tightness_log"]
feature_cols = [
    "tenure_log",
    "expense_log",
    "degree_log",
    "tightness_log",
    "entropy",
    "chgexpense",
    "chgdegree",
] # 最终使用的特征列
sample_data[log_cols] = np.log(sample_data[log_cols_o])
pred_data[log_cols] = np.log(pred_data[log_cols_o])

In [ ]:
X_train = (sample_data[feature_cols] - sample_data[feature_cols].mean()) / sample_data[feature_cols].std()
# 测试集也使用训练集的均值和标准差进行标准化
X_pred = (pred_data[feature_cols] - sample_data[feature_cols].mean()) / sample_data[feature_cols].std()
y_train = sample_data["churn"]
y_pred = pred_data["churn"]

In [ ]:
# 为得到更详细的回归结果，使用 statsmodels 进行拟合
model = sm.Logit(y_train, sm.add_constant(X_train)).fit() # 使用 statsmodels 的 logistic 模型进行含截距项的拟合
print(model.summary())

#### 系数估计结果解读

- const: 截距项，系数为-4.8350，表示在其他变量均为0时，用户流失的对数几率(log odds)为-4.8350。
- tenure_log: 在网时长的对数值，系数为-0.1306，表示在网时长的对数值每增加1个单位，流失的对数几率减少约0.1306，说明在网时长越长，用户越不容易流失。
- expense_log: 当月花费的对数值，系数为-0.2256，表示当月花费的对数值每增加1个单位，流失的对数几率减少约0.2256，说明当月花费越高，用户越不容易流失。
- degree_log: 个体的度的对数值，系数为-0.7172，表示个体的度的对数值每增加1个单位，流失的对数几率减少约0.7172，说明个体的度越高，用户越不容易流失。
- tightness_log: 联系强度的对数值，系数为-0.1051，表示联系强度的对数值每增加1个单位，流失的对数几率减少约0.1051，说明联系强度越高，用户越不容易流失。
- entropy: 个体信息熵，系数为0.0771，表示个体信息熵每增加1个单位，流失的对数几率增加约0.0771，说明个体信息熵越高，用户越容易流失，但p值为0.446较大，影响不显著。
- chgexpense: 花费的变化，系数为-0.1387，表示花费的变化每增加1个单位，流失的对数几率减少约0.1387，说明花费的变化越高，用户越不容易流失。
- chgdegree: 个体度的变化，系数为-0.2729，表示个体度的变化每增加1个单位，流失的对数几率减少约0.2729，说明个体度的变化越高，用户越不容易流失。

除 entropy 外其他变量的p值都小于0.01，具有显著性。

In [ ]:
# 使用模型预测概率
prob_pred = model.predict(sm.add_constant(X_pred))
prob_sample = model.predict(sm.add_constant(X_train))
# 使用sklearn.metrics.RocCurveDisplay绘制 ROC 曲线
roc_sample = RocCurveDisplay.from_predictions(y_train,prob_sample,name='sample')
roc_pred = RocCurveDisplay.from_predictions(y_pred,prob_pred,name='pred')

In [ ]:
# 使用 sklearn.metrics.roc_auc_score 计算 AUC
auc_sample = roc_auc_score(y_train, prob_sample)
auc_pred = roc_auc_score(y_pred, prob_pred)
print(f"AUC of sample_log_std data: {auc_sample}")
print(f"AUC of pred_log_std data: {auc_pred}")

训练集与测试集ROC曲线向左上角凸起，且形状基本一致，AUC分别为为0.775/0.787，拟合表现良好，泛化能力优秀。